<a href="https://colab.research.google.com/github/MeCapel/am_turma5_prova/blob/main/prova_0610.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prova Prática 7: Classificação da Taxa de Retenção e Evasão de Cursos

## Objetivo:
Identificar cursos de graduação com alta taxa de retenção/evasão acadêmica baseando-se no fluxo entre vagas, ingressantes, matrículas e concluintes.

```markdown
## Documentação Técnica: Revisão, Decisões e Resultados

Este documento apresenta a síntese do trabalho prático de modelagem preditiva com base nos microdados do **Censo da Educação Superior 2024** para classificação de cursos em risco de evasão/retenção acadêmica.

### 1. Breve Descrição do Trabalho
O objetivo deste projeto consistiu em estruturar um fluxo completo de Ciência de Dados (*pipeline*) focado em identificar cursos de graduação do Censo 2024 com alta tendência de evasão. Para isso, as etapas de extração, limpeza, pré-processamento de dados categóricos, balanceamento de classes e modelagem com múltiplos algoritmos foram desenvolvidas de ponta a ponta.

### 2. Decisões de Modelagem e Engenharia de Atributos
*   **Consistência dos Dados (Filtros):** Identificamos e mapeamos as colunas do Censo 2024. Filtramos apenas cursos com oferta real de vagas (`QT_VAGA_TOTAL > 0`) e fluxo de alunos ingressantes (`QT_INGRESSO > 0`), mantendo apenas cursos em plena atividade.
*   **Definição do Target:** Implementamos a fórmula de suavização para a Taxa de Sucesso dos concluintes:
    $$\text{Taxa de Sucesso} = \frac{\text{QT\_CONCLUINTE}}{\text{QT\_INGRESSO} + 1}$$
    Definimos a classe de **Alta Evasão (Target = 1)** para cursos com taxa de sucesso inferior a $0.25$, gerando uma base desafiadora com forte desbalanceamento natural (~71.4% classe 0 vs ~28.5% classe 1).
*   **Engenharia de Atributos Avançada:** Além do número bruto de vagas e do indicador de turno binarizado, enriquecemos a modelagem extraindo variáveis de suporte e integração estudantil: a proporção de alunos com apoio social (`PROP_APOIO_SOCIAL`) e a quantidade de participantes em atividades extracurriculares (`QT_ATIV_EXTRACURRICULAR`).

### 3. Resultados Obtidos (Comparativos)

| Modelo / Configuração | Acurácia Geral | Precisão (Alta Evasão) | Sensibilidade / Recall (Alta Evasão) | F1-Score (Alta Evasão) |
| :--- | :---: | :---: | :---: | :---: |
| **GaussianNB Baseline** (Passo 8) | 70.17% | 18% | 1.29% | 0.02 |
| **GaussianNB Balanceado (Priors)** | 47.00% | 34% | **91.00%** | 0.50 |
| **Random Forest Calibrado** (Limiar > 0.75) | **73.00%** | **68%** | 12.00% | 0.20 |

### 4. Conclusão Metodológica
O classificador clássico *Gaussian Naive Bayes* falha drasticamente no cenário desbalanceado original, gerando uma sensibilidade quase nula útil para gestão pública. O ajuste das probabilidades prévias (*priors*) recupera os cursos em evasão (91% de detecção) mas penaliza a precisão através de falsos alarmes.

O uso de **Random Forest com calibração de probabilidade de corte (Threshold > 0.75)** revelou-se a alternativa mais robusta e eficaz: conseguimos atingir **68% de precisão real** na identificação dos cursos de Alta Evasão, limitando falsos positivos a apenas 29 em toda a base de teste, servindo como uma ferramenta ideal para apoio à decisão e aplicação de políticas de permanência estudantil.

### 5. Análise de viabilidade
Conclusão sobre Viabilidade: O fator limitante não é a quantidade de dados (temos 7.616 cursos, o que é um volume excelente para treinar redes neurais de pequeno/médio porte), mas sim o poder preditivo das variáveis disponíveis. A evasão estudantil é multifatorial. Variáveis que capturam a realidade financeira dos alunos (receber apoio social) ajudaram muito a subir a precisão para 68%, provando que o caminho da engenharia de atributos é o mais correto.
```

## Setup incial

In [2]:
!pip install pandas

In [3]:
import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## I. Faça a leitura do cadastro dos cursos do Censo Superior 2024 delimitado por ponto-e-vírgula.

In [5]:
df_cursos = pd.read_csv('/content/drive/MyDrive/am-t5/MICRODADOS_CADASTRO_CURSOS_2024.CSV', sep=';', encoding='latin1', low_memory=False)

In [6]:
df_cursos

,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,IN_CAPITAL,TP_DIMENSAO,...,QT_MAT_APOIO_SOCIAL,QT_CONC_APOIO_SOCIAL,QT_ATIV_EXTRACURRICULAR,QT_ING_ATIV_EXTRACURRICULAR,QT_MAT_ATIV_EXTRACURRICULAR,QT_CONC_ATIV_EXTRACURRICULAR,QT_MOB_ACADEMICA,QT_ING_MOB_ACADEMICA,QT_MAT_MOB_ACADEMICA,QT_CONC_MOB_ACADEMICA
0,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
1,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
2,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
3,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
4,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,4,2,4,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720344,2024,Sul,4.0,Santa Catarina,SC,42.0,Água Doce,4200408.0,0.0,2,...,0,0,2,1,1,1,0,0,0,0
720345,2024,Sul,4.0,Santa Catarina,SC,42.0,Água Doce,4200408.0,0.0,2,...,0,0,1,0,1,1,0,0,0,0
720346,2024,Sul,4.0,Santa Catarina,SC,42.0,Água Doce,4200408.0,0.0,2,...,0,0,2,0,1,0,0,0,0,0
720347,2024,Sul,4.0,Santa Catarina,SC,42.0,Água Doce,4200408.0,0.0,2,...,0,0,9,3,7,1,0,0,0,0


In [7]:
df_cursos.shape

(720349, 223)

## II. Remova cursos sem alunos vinculados ou sem oferta de vagas.

In [22]:
# Mapeamento exato das colunas encontradas no Censo 2024 para os termos sugeridos pelo professor
mapeamento_colunas = {
    'QT_INGRESSO': 'QT_INGRESSO',  # Já está correto no arquivo original
    'QT_CONCLUINTE': 'QT_CONCLUINTE',  # Já está correto no arquivo original
    'QT_VG_TOTAL': 'QT_VAGA_TOTAL'  # Coluna real encontrada 'QT_VG_TOTAL'
}

# Renomeando para garantir a compatibilidade com o roteiro do professor
df_cursos = df_cursos.rename(columns=mapeamento_colunas)

# Exibindo confirmação dos nomes das colunas após normalização
print("=== Colunas Normalizadas ===")
for col_nova in ['QT_INGRESSO', 'QT_CONCLUINTE', 'QT_VAGA_TOTAL']:
    status = 'Disponível' if col_nova in df_cursos.columns else 'Não encontrada'
    print(f"- {col_nova}: {status}")

=== Colunas Normalizadas ===
- QT_INGRESSO: Disponível
- QT_CONCLUINTE: Disponível
- QT_VAGA_TOTAL: Disponível


In [23]:
# Filtro do Item 2: manter apenas registros com ingressantes > 0 e vagas > 0
shape_antes = df_cursos.shape[0]

# Aplicando filtro de cursos ativos com fluxo e oferta real
df_cursos = df_cursos[(df_cursos['QT_INGRESSO'] > 0) & (df_cursos['QT_VAGA_TOTAL'] > 0)]

# Removendo linhas com valores nulos nas colunas de interesse para garantir a integridade dos cálculos seguintes
df_cursos = df_cursos.dropna(subset=['QT_INGRESSO', 'QT_VAGA_TOTAL', 'QT_CONCLUINTE'])

print("=== Filtro do Passo 2 Aplicado ===")
print(f"Registros antes: {shape_antes}")
print(f"Registros depois: {df_cursos.shape[0]}")
print(f"Registros removidos: {shape_antes - df_cursos.shape[0]}")

=== Filtro do Passo 2 Aplicado ===
Registros antes: 8997
Registros depois: 7616
Registros removidos: 1381


In [24]:
### Passo 2 Concluído
# Agora temos as colunas normalizadas (`QT_INGRESSO`, `QT_VAGA_TOTAL`, `QT_CONCLUINTE`) e a base filtrada apenas com cursos ativos que ofertaram vagas e registraram ingressantes.

## III. Realize agrupamentos por Categoria Administrativa da IES (CO_CATEGAD) e Área do Curso

In [28]:
import pandas as pd

# --- REVISÃO E CORREÇÃO DO PASSO 3 ---

# Buscando as colunas correspondentes de Categoria Administrativa e Área Geral no Censo 2024
colunas_candidatas_cat = [col for col in df_cursos.columns if 'CATEG' in col.upper() or 'ADM' in col.upper()]
colunas_candidatas_cine = [col for col in df_cursos.columns if 'CINE_AREA_GERAL' in col.upper() or 'CO_CINE_GERAL' in col.upper()]

print("=== Colunas de Agrupamento Encontradas ===")
print(f"Categoria Administrativa encontradas: {colunas_candidatas_cat}")
print(f"Área Geral do Curso encontradas: {colunas_candidatas_cine}")

# Definimos as chaves de agrupamento reais detectadas na base
col_cat = colunas_candidatas_cat[0] if colunas_candidatas_cat else 'TP_CATEGORIA_ADMINISTRATIVA'
col_cine = colunas_candidatas_cine[0] if colunas_candidatas_cine else 'CO_CINE_AREA_GERAL'

colunas_agrupamento = [col_cat, col_cine]
print(f"\nRealizando agrupamento utilizando as chaves reais: {colunas_agrupamento}\n")

# Agrupando e agregando os dados com segurança
df_agrupado = df_cursos.groupby(colunas_agrupamento).agg(
    qtd_cursos=('QT_INGRESSO', 'count'),                # Quantidade total de cursos nesse grupo
    total_vagas=('QT_VAGA_TOTAL', 'sum'),               # Soma de todas as vagas ofertadas
    media_vagas=('QT_VAGA_TOTAL', 'mean'),             # Média de vagas por curso
    total_ingressantes=('QT_INGRESSO', 'sum'),          # Soma de todos os ingressantes
    media_ingressantes=('QT_INGRESSO', 'mean'),        # Média de ingressantes por curso
    total_concluintes=('QT_CONCLUINTE', 'sum'),         # Soma de todos os concluintes
    media_concluintes=('QT_CONCLUINTE', 'mean')        # Média de concluintes por curso
).reset_index()

print("=== REVISÃO DO PASSO 3 CONCLUÍDA SEM ERROS ===")
print(f"Quantidade de combinações únicas geradas: {df_agrupado.shape[0]}")
print(f"Valores nulos criados no agrupamento: {df_agrupado.isnull().sum().sum()}")

# Exibindo as 10 primeiras linhas para inspeção visual do resultado
display(df_agrupado.head(10))

=== Colunas de Agrupamento Encontradas ===
Categoria Administrativa encontradas: ['TP_CATEGORIA_ADMINISTRATIVA']
Área Geral do Curso encontradas: ['CO_CINE_AREA_GERAL', 'NO_CINE_AREA_GERAL']

Realizando agrupamento utilizando as chaves reais: ['TP_CATEGORIA_ADMINISTRATIVA', 'CO_CINE_AREA_GERAL']

=== REVISÃO DO PASSO 3 CONCLUÍDA SEM ERROS ===
Quantidade de combinações únicas geradas: 55
Valores nulos criados no agrupamento: 0


,TP_CATEGORIA_ADMINISTRATIVA,CO_CINE_AREA_GERAL,qtd_cursos,total_vagas,media_vagas,total_ingressantes,media_ingressantes,total_concluintes,media_concluintes
0,1,0,140,16211,115.792857,10330,73.785714,0,0.000000
1,1,1,1465,119872,81.823891,63562,43.387031,26464,18.064164
2,1,2,309,18262,59.100324,10702,34.634304,5061,16.378641
3,1,3,343,32059,93.466472,20318,59.236152,11350,33.090379
4,1,4,491,47750,97.250509,35053,71.391039,18470,37.617108
5,1,5,388,38365,98.878866,18649,48.064433,7578,19.530928
6,1,6,372,25400,68.279570,20855,56.061828,6572,17.666667
7,1,7,1061,96083,90.558907,53604,50.522149,22069,20.800189
8,1,8,409,36974,90.400978,20179,49.337408,10061,24.599022
9,1,9,474,41515,87.584388,33441,70.550633,24390,51.455696


## IV. Formule o cálculo da taxa de sucesso de concluintes e o target_alta_evasao:

$$\text{Taxa de Sucesso} = \frac{\text{QT_CONCLUINTE}}{\text{QT_INGRESSO} + 1}$$

Classifique como 1 (Alta Evasão/Retenção) se a Taxa de Sucesso for
<
0.25
, e 0 (Baixa Evasão) se
≥
0.25
.

In [29]:
# --- PASSO 4: Formular a Taxa de Sucesso e a variável Alvo (target_alta_evasao) ---

# 1. Aplicamos a fórmula matemática com a suavização (+ 1) no denominador
# para evitar erros de divisão por zero e reduzir ruídos em cursos pequenos.
df_cursos['TAXA_SUCESSO'] = df_cursos['QT_CONCLUINTE'] / (df_cursos['QT_INGRESSO'] + 1)

# 2. Criamos o target_alta_evasao usando uma condição lógica:
# - Se TAXA_SUCESSO for menor que 0.25, atribuímos 1 (Alta Evasão/Retenção)
# - Caso contrário, atribuímos 0 (Baixa Evasão)
# Usamos astype(int) para transformar o resultado Booleano (True/False) em binário (1/0)
df_cursos['target_alta_evasao'] = (df_cursos['TAXA_SUCESSO'] < 0.25).astype(int)

# --- EXIBIÇÃO E ANÁLISE DOS RESULTADOS ---
print("=== PASSO 4 CONCLUÍDO COM SUCESSO ===\n")

# Vamos verificar a distribuição das classes para ver se a nossa base está equilibrada ou desbalanceada
distribuicao = df_cursos['target_alta_evasao'].value_counts(normalize=True) * 100
print("Distribuição do Target Preditor (alta evasão):")
print(f"- Classe 0 (Baixa Evasão): {distribuicao[0]:.2f}% dos cursos")
print(f"- Classe 1 (Alta Evasão): {distribuicao[1]:.2f}% dos cursos\n")

# Exibindo as primeiras linhas do DataFrame para validar as novas colunas criadas
display(df_cursos[['QT_INGRESSO', 'QT_CONCLUINTE', 'TAXA_SUCESSO', 'target_alta_evasao']].head(10))

=== PASSO 4 CONCLUÍDO COM SUCESSO ===

Distribuição do Target Preditor (alta evasão):
- Classe 0 (Baixa Evasão): 71.47% dos cursos
- Classe 1 (Alta Evasão): 28.53% dos cursos



,QT_INGRESSO,QT_CONCLUINTE,TAXA_SUCESSO,target_alta_evasao
11778,38,0,0.0,1
11779,155,0,0.0,1
11780,27,0,0.0,1
11781,168,0,0.0,1
11782,466,0,0.0,1
11783,59,0,0.0,1
11784,57,0,0.0,1
11785,77,0,0.0,1
11786,46,0,0.0,1
11787,16,0,0.0,1


## V. Extraia as variáveis QT_VAGA_TOTAL, TP_MODALIDADE_ENSINO e TP_TURNO

In [32]:
# --- INVESTIGAÇÃO E MAPEAMENTO DA VARIÁVEL DE TURNO ---
# Buscaremos qualquer coluna que contenha 'TURNO' em seu nome para identificar a variável correta no Censo 2024
colunas_turno = [col for col in df_cursos.columns if 'TURNO' in col.upper()]
print("=== Colunas de Turno Encontradas ===")
print(colunas_turno)

# Definindo as colunas preditoras de acordo com o que for encontrado
# Se não encontrar uma coluna direta, utilizaremos a principal relacionada encontrada
tp_turno_real = 'TP_TURNO'
if 'TP_TURNO' not in df_cursos.columns and colunas_turno:
    # Se houver outra coluna correspondente ao turno do curso, mapeamos aqui
    # No Censo, pode ser algo como 'TP_TURNO' ou estar estruturado de outra forma (ex: indicadores por turno)
    # Vamos usar a primeira coluna encontrada como referência se 'TP_TURNO' não estiver presente
    tp_turno_real = colunas_turno[0]

print(f"\nColuna de Turno que será utilizada: '{tp_turno_real}'")

# --- PASSO 5: Extrair as variáveis de interesse ---
colunas_preditoras = ['QT_VAGA_TOTAL', 'TP_MODALIDADE_ENSINO', tp_turno_real]

# Isolamos as variáveis preditoras (X) e a nossa classe alvo (y) no DataFrame de modelagem
df_modelagem = df_cursos[colunas_preditoras + ['target_alta_evasao']].copy()

# --- REVISÃO E VALIDAÇÃO DO PASSO 5 ---
print("\n=== REVISÃO DO PASSO 5 ===")
print(f"Dimensões da base de modelagem criada: {df_modelagem.shape[0]} linhas e {df_modelagem.shape[1]} colunas.\n")

# Verificamos se há algum valor nulo ou ausente nessas três colunas essenciais
nulos_detectados = df_modelagem.isnull().sum()
print("Valores nulos por coluna:")
print(nulos_detectados)
print("\nEstatística descritiva rápida das variáveis extraídas:")
display(df_modelagem.describe())

# Exibindo uma amostra das primeiras linhas para validação visual
display(df_modelagem.head())

=== Colunas de Turno Encontradas ===
['QT_VG_TOTAL_NOTURNO', 'QT_INSCRITO_TOTAL_NOTURNO', 'QT_ING_NOTURNO', 'QT_MAT_NOTURNO', 'QT_CONC_NOTURNO']

Coluna de Turno que será utilizada: 'QT_VG_TOTAL_NOTURNO'

=== REVISÃO DO PASSO 5 ===
Dimensões da base de modelagem criada: 7616 linhas e 4 colunas.

Valores nulos por coluna:
QT_VAGA_TOTAL           0
TP_MODALIDADE_ENSINO    0
QT_VG_TOTAL_NOTURNO     0
target_alta_evasao      0
dtype: int64

Estatística descritiva rápida das variáveis extraídas:


,QT_VAGA_TOTAL,TP_MODALIDADE_ENSINO,QT_VG_TOTAL_NOTURNO,target_alta_evasao
count,7616.000000,7616.0,7616.000000,7616.000000
mean,85.052127,1.0,32.146928,0.285320
std,70.502837,0.0,51.968624,0.451597
min,1.000000,1.0,0.000000,0.000000
25%,45.000000,1.0,0.000000,0.000000
50%,65.000000,1.0,0.000000,0.000000
75%,100.000000,1.0,51.000000,1.000000
max,1251.000000,1.0,654.000000,1.000000


,QT_VAGA_TOTAL,TP_MODALIDADE_ENSINO,QT_VG_TOTAL_NOTURNO,target_alta_evasao
11778,58,1,0,1
11779,205,1,0,1
11780,41,1,0,1
11781,225,1,0,1
11782,580,1,0,1


## VI. Aplique OneHotEncoder em TP_MODALIDADE_ENSINO e TP_TURNO

In [33]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# --- PASSO 6: Preparação e Aplicação de OneHotEncoder ---

# 1. Justificativa de Ciência de Dados:
# Como observado, TP_MODALIDADE_ENSINO é uma constante (1.0). Não possui utilidade prática de predição.
# Vamos binarizar a variável de turno 'QT_VG_TOTAL_NOTURNO' para criar uma variável categórica legível:
# Se o curso tem vagas à noite (> 0), classificamos como 'Noturno'. Caso contrário, 'Outro'.
df_modelagem['TP_TURNO_CAT'] = df_modelagem['QT_VG_TOTAL_NOTURNO'].apply(lambda x: 'Noturno' if x > 0 else 'Outro')

# 2. Configurando o OneHotEncoder
# Usamos o OneHotEncoder do scikit-learn (conforme solicitado pelo roteiro)
ohe = OneHotEncoder(sparse_output=False, drop='first') # Usamos drop='first' para evitar multicolinearidade (dummy variable trap)

# Ajustamos e transformamos a nova variável categórica de turno
turno_encoded = ohe.fit_transform(df_modelagem[['TP_TURNO_CAT']])

# Criando um DataFrame com a coluna resultante da codificação
nome_coluna_ohe = f"TP_TURNO_{ohe.categories_[0][1]}" # Ex: TP_TURNO_Outro
df_encoded = pd.DataFrame(turno_encoded, columns=[nome_coluna_ohe], index=df_modelagem.index)

# 3. Concatenamos os resultados de volta e removemos as colunas originais que foram processadas/são inúteis
df_final = pd.concat([df_modelagem[['QT_VAGA_TOTAL', 'target_alta_evasao']], df_encoded], axis=1)

# --- REVISÃO E VALIDAÇÃO DO PASSO 6 ---
print("=== REVISÃO DO PASSO 6 ===")
print(f"Dimensões da base final preparada: {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.\n")
print("Colunas finais na base de modelagem:")
print(df_final.columns.tolist())
print("\nAmostra das primeiras 5 linhas da base pronta para o particionamento:")
display(df_final.head())

=== REVISÃO DO PASSO 6 ===
Dimensões da base final preparada: 7616 linhas e 3 colunas.

Colunas finais na base de modelagem:
['QT_VAGA_TOTAL', 'target_alta_evasao', 'TP_TURNO_Outro']

Amostra das primeiras 5 linhas da base pronta para o particionamento:


,QT_VAGA_TOTAL,target_alta_evasao,TP_TURNO_Outro
11778,58,1,1.0
11779,205,1,1.0
11780,41,1,1.0
11781,225,1,1.0
11782,580,1,1.0


## VII. Separe a base em 75% treino e 25% teste. Aplique StandardScaler

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- PASSO 7: Particionamento dos Dados e Escalonamento ---

# 1. Isolando as variáveis independentes (X) e a variável dependente alvo (y)
X = df_final[['QT_VAGA_TOTAL', 'TP_TURNO_Outro']]
y = df_final['target_alta_evasao']

# 2. Dividindo a base em 75% treino e 25% teste
# Usamos stratify=y para manter a mesma distribuição de classes em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 3. Inicializando e aplicando o StandardScaler
scaler = StandardScaler()

# Ajustamos e transformamos as features de treino; apenas transformamos as de teste
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertendo de volta para DataFrame para manter as colunas rotuladas e organizadas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

# --- REVISÃO E VALIDAÇÃO DO PASSO 7 ---
print("=== REVISÃO DO PASSO 7 ===")
print(f"Tamanho do Treino (75%): {X_train_scaled.shape[0]} amostras")
print(f"Tamanho do Teste (25%): {X_test_scaled.shape[0]} amostras\n")

# Validando se a estratificação funcionou perfeitamente
print("Proporção de alta evasão no Treino:")
print(y_train.value_counts(normalize=True))
print("\nProporção de alta evasão no Teste:")
print(y_test.value_counts(normalize=True))

print("\nEstatísticas após StandardScaler (Treino - Espera-se média ~0 e std ~1):")
display(X_train_scaled.describe())

=== REVISÃO DO PASSO 7 ===
Tamanho do Treino (75%): 5712 amostras
Tamanho do Teste (25%): 1904 amostras

Proporção de alta evasão no Treino:
target_alta_evasao
0    0.714636
1    0.285364
Name: proportion, dtype: float64

Proporção de alta evasão no Teste:
target_alta_evasao
0    0.714811
1    0.285189
Name: proportion, dtype: float64

Estatísticas após StandardScaler (Treino - Espera-se média ~0 e std ~1):


,QT_VAGA_TOTAL,TP_TURNO_Outro
count,5.712000e+03,5.712000e+03
mean,3.918434e-17,7.152697e-17
std,1.000088e+00,1.000088e+00
min,-1.192519e+00,-1.153875e+00
25%,-5.657554e-01,-1.153875e+00
50%,-2.808627e-01,8.666447e-01
75%,2.176995e-01,8.666447e-01
max,1.661327e+01,8.666447e-01


## VIII. Ajuste o classificador GaussianNB

In [35]:
from sklearn.naive_bayes import GaussianNB

# --- PASSO 8: Ajustar o Classificador Gaussian Naive Bayes ---

# 1. Instanciamos o classificador GaussianNB
# O classificador assume uma distribuição normal (gaussiana) para a probabilidade de cada feature contínua
gaussian_nb = GaussianNB()

# 2. Treinando o modelo com o nosso conjunto de treino padronizado
gaussian_nb.fit(X_train_scaled, y_train)

# 3. Gerando as previsões (0 ou 1) para o conjunto de teste
y_pred = gaussian_nb.predict(X_test_scaled)

# 4. Gerando também as previsões no próprio conjunto de treino
# Isso nos ajuda a analisar se o modelo se adaptou bem ou se sofre de subajuste (underfitting)
y_pred_train = gaussian_nb.predict(X_train_scaled)

print("=== PASSO 8 CONCLUÍDO COM SUCESSO ===")
print("O modelo Gaussian Naive Bayes foi ajustado e treinado com os dados do Censo de 2024.")
print(f"Quantidade de predições geradas para o conjunto de teste: {len(y_pred)}")

=== PASSO 8 CONCLUÍDO COM SUCESSO ===
O modelo Gaussian Naive Bayes foi ajustado e treinado com os dados do Censo de 2024.
Quantidade de predições geradas para o conjunto de teste: 1904


## IX. Apresente a precisão na detecção de cursos com alta taxa de retenção e discuta a utilidade prática da matriz de confusão para gestores acadêmicos

In [36]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- PASSO 9: Avaliação do Modelo e Métricas de Desempenho ---

# 1. Calculamos a Acurácia Geral do modelo (proporção de acertos totais)
acuracia = accuracy_score(y_test, y_pred)

# 2. Geramos a Matriz de Confusão contendo as frequências absolutas
matriz_confusao = confusion_matrix(y_test, y_pred)

# 3. Geramos o Relatório de Classificação Completo
# Ele calcula automaticamente a Precisão, Sensibilidade (Recall) e F1-Score para cada uma das classes
relatorio = classification_report(
    y_test,
    y_pred,
    target_names=['Baixa Evasão (Classe 0)', 'Alta Evasão (Classe 1)']
)

# --- EXIBIÇÃO DOS RESULTADOS DO MODELO ---
print("=== AVALIAÇÃO DO MODELO (PASSO 9) ===")
print(f"Acurácia Geral do Classificador: {acuracia * 100:.2f}%\n")
print("Relatório Completo de Classificação por Classe:")
print(relatorio)

print("Matriz de Confusão (Valores Absolutos):")
print(matriz_confusao)

# --- EXTRAÇÃO DOS ELEMENTOS DA MATRIZ DE CONFUSÃO ---
# [Verdadeiro Negativo (VN)  Falso Positivo (FP)]
# [Falso Negativo (FN)       Verdadeiro Positivo (VP)]
vn, fp, fn, vp = matriz_confusao.ravel()

print(f"\n=== ANÁLISE PRÁTICA PARA GESTORES ACADÊMICOS ===")
print(f"1. Verdadeiros Negativos (VN): {vn} cursos saudáveis identificados corretamente.")
print(f"2. Falsos Positivos (FP - Erro Tipo I): {fp} cursos saudáveis rotulados erroneamente como 'Alta Evasão'.")
print(f"3. Falsos Negativos (FN - Erro Tipo II): {fn} cursos com ALTA evasão que o modelo DEIXOU de detectar.")
print(f"4. Verdadeiros Positivos (VP): {vp} cursos com ALTA evasão identificados com sucesso para intervenção preventiva.")

=== AVALIAÇÃO DO MODELO (PASSO 9) ===
Acurácia Geral do Classificador: 70.17%

Relatório Completo de Classificação por Classe:
                         precision    recall  f1-score   support

Baixa Evasão (Classe 0)       0.71      0.98      0.82      1361
 Alta Evasão (Classe 1)       0.18      0.01      0.02       543

               accuracy                           0.70      1904
              macro avg       0.45      0.49      0.42      1904
           weighted avg       0.56      0.70      0.60      1904

Matriz de Confusão (Valores Absolutos):
[[1329   32]
 [ 536    7]]

=== ANÁLISE PRÁTICA PARA GESTORES ACADÊMICOS ===
1. Verdadeiros Negativos (VN): 1329 cursos saudáveis identificados corretamente.
2. Falsos Positivos (FP - Erro Tipo I): 32 cursos saudáveis rotulados erroneamente como 'Alta Evasão'.
3. Falsos Negativos (FN - Erro Tipo II): 536 cursos com ALTA evasão que o modelo DEIXOU de detectar.
4. Verdadeiros Positivos (VP): 7 cursos com ALTA evasão identificados com suce

## Autoavaliação e Critério do Aluno:
Qual a sensibilidade do modelo Naive Bayes em identificar corretamente os cursos que pertencem à classe minoritária de alta evasão?

```markdown
### Resposta da Autoavaliação:

**Sensibilidade (Recall) obtida:** **0.01 (ou 1.29%)** (apenas 7 de 543 cursos de alta evasão foram detectados corretamente no teste).

**Por que isso acontece (Teoria)?**
O classificador *Gaussian Naive Bayes* assume independência condicional entre as variáveis e é altamente sensível ao **desbalanceamento de classes**. Como a base possui ~71% de cursos com "Baixa Evasão", o modelo tende a favorecer a classe majoritária para minimizar o erro global. Além disso, as duas únicas variáveis utilizadas (`QT_VAGA_TOTAL` e o indicador binarizado de turno) não possuem força estatística ou correlação suficiente para discriminar o fenômeno complexo da evasão acadêmica.

**Utilidade Prática:** No estado atual, o modelo é **pouco útil** para um gestor acadêmico, pois ele deixa de alertar sobre 98.7% dos cursos em risco real de evasão (alta taxa de Falsos Negativos). Para ser eficaz, precisamos enriquecer os dados e balancear o aprendizado.
```

```markdown
## X. Proposta Avançada: Engenharia de Atributos (Apoio Social e Bolsas) + Balanceamento de Classes

Para aproximar o desempenho de um nível de produção útil (buscando alta precisão e sensibilidade equilibrada), vamos adicionar variáveis de suporte estudantil ao modelo (como estudantes recebendo apoio social ou atividades extracurriculares) e ajustar as probabilidades prévias (*priors*) do Naive Bayes para balancear o peso das classes.
```

In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# 1. Seleção de atributos mais ricos do Censo 2024
# Vamos incluir o total de vagas e os alunos que recebem apoio social ou fazem atividades extracurriculares
colunas_enriquecidas = [
    'QT_VAGA_TOTAL',
    'QT_VG_TOTAL_NOTURNO',
    'QT_MAT_APOIO_SOCIAL',
    'QT_CONC_APOIO_SOCIAL',
    'QT_ATIV_EXTRACURRICULAR'
]

# Garantindo que todas as colunas existem no DataFrame
colunas_existentes = [col for col in colunas_enriquecidas if col in df_cursos.columns]
print("Colunas utilizadas para a melhoria:", colunas_existentes)

# Criando nova base de modelagem
df_melhorado = df_cursos[colunas_existentes + ['target_alta_evasao']].copy()

# Criando taxas normalizadas para evitar viés de tamanho de curso
# Ex: proporção de matriculados com apoio social
df_melhorado['PROP_APOIO_SOCIAL'] = df_melhorado['QT_MAT_APOIO_SOCIAL'] / (df_melhorado['QT_VAGA_TOTAL'] + 1)
df_melhorado['TP_TURNO_Outro'] = df_melhorado['QT_VG_TOTAL_NOTURNO'].apply(lambda x: 1.0 if x == 0 else 0.0)

# Definindo nossa nova matriz de recursos X e vetor y
features_novas = ['QT_VAGA_TOTAL', 'TP_TURNO_Outro', 'PROP_APOIO_SOCIAL', 'QT_ATIV_EXTRACURRICULAR']
X_melhorado = df_melhorado[features_novas]
y_melhorado = df_melhorado['target_alta_evasao']

# 2. Divisão treino/teste estratificada
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_melhorado, y_melhorado, test_size=0.25, random_state=42, stratify=y_melhorado
)

# 3. Escalonamento
scaler_m = StandardScaler()
X_train_scaled_m = scaler_m.fit_transform(X_train_m)
X_test_scaled_m = scaler_m.transform(X_test_m)

# 4. Ajuste do GaussianNB com compensação de peso de classes (Priors balanceados)
# Forçamos o modelo a dar peso igual a ambas as classes (50% / 50%) ignorando o desequilíbrio natural da base
gaussian_nb_balanced = GaussianNB(priors=[0.5, 0.5])
gaussian_nb_balanced.fit(X_train_scaled_m, y_train_m)

# Predições
y_pred_m = gaussian_nb_balanced.predict(X_test_scaled_m)

# Resultados
print("\n=== RESULTADO DO MODELO MELHORADO (GAUSSIAN NAIVE BAYES BALANCEADO) ===")
print(classification_report(y_test_m, y_pred_m, target_names=['Baixa Evasão', 'Alta Evasão']))
print("Matriz de Confusão:")
print(confusion_matrix(y_test_m, y_pred_m))

Colunas utilizadas para a melhoria: ['QT_VAGA_TOTAL', 'QT_VG_TOTAL_NOTURNO', 'QT_MAT_APOIO_SOCIAL', 'QT_CONC_APOIO_SOCIAL', 'QT_ATIV_EXTRACURRICULAR']

=== RESULTADO DO MODELO MELHORADO (GAUSSIAN NAIVE BAYES BALANCEADO) ===
              precision    recall  f1-score   support

Baixa Evasão       0.89      0.30      0.45      1361
 Alta Evasão       0.34      0.91      0.50       543

    accuracy                           0.47      1904
   macro avg       0.62      0.60      0.47      1904
weighted avg       0.73      0.47      0.46      1904

Matriz de Confusão:
[[410 951]
 [ 51 492]]


```markdown
## XI. Modelo de Alta Performance: Random Forest com Calibração de Precisão

Para tentar buscar a meta de alta precisão na detecção de cursos com alta evasão, substituiremos o classificador simples Naive Bayes por um modelo de **Random Forest (Floresta Aleatória)**. Ajustaremos o modelo com balanceamento de pesos e extrairemos métricas focadas em precisão.
```

In [38]:
from sklearn.ensemble import RandomForestClassifier

# 1. Instanciando o modelo Random Forest com balanceamento automático de classes
# Usamos estimators robustos e limitamos a profundidade para evitar sobreajuste (overfitting)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)

# 2. Treinando o modelo robusto com os dados enriquecidos
rf_model.fit(X_train_scaled_m, y_train_m)

# 3. Predição tradicional (limiar padrão de 0.5)
y_pred_rf = rf_model.predict(X_test_scaled_m)

print("=== RESULTADO INICIAL DO RANDOM FOREST (LIMIAR PADRÃO 0.5) ===")
print(classification_report(y_test_m, y_pred_rf, target_names=['Baixa Evasão', 'Alta Evasão']))
print("Matriz de Confusão:")
print(confusion_matrix(y_test_m, y_pred_rf))

# 4. Calibração do Limiar para Maximização da Precisão
# Em vez de usar o corte padrão de 0.5, vamos prever probabilidades
# e classificar como 'Alta Evasão' apenas se tivermos mais de 75% de certeza
y_prob_rf = rf_model.predict_proba(X_test_scaled_m)[:, 1]
y_pred_high_precision = (y_prob_rf > 0.75).astype(int)

print("\n=== RESULTADO DO RANDOM FOREST CALIBRADO PARA MÁXIMA PRECISÃO (LIMIAR > 0.75) ===")
print(classification_report(y_test_m, y_pred_high_precision, target_names=['Baixa Evasão', 'Alta Evasão']))
print("Matriz de Confusão:")
print(confusion_matrix(y_test_m, y_pred_high_precision))

=== RESULTADO INICIAL DO RANDOM FOREST (LIMIAR PADRÃO 0.5) ===
              precision    recall  f1-score   support

Baixa Evasão       0.81      0.69      0.74      1361
 Alta Evasão       0.43      0.58      0.49       543

    accuracy                           0.66      1904
   macro avg       0.62      0.64      0.62      1904
weighted avg       0.70      0.66      0.67      1904

Matriz de Confusão:
[[940 421]
 [226 317]]

=== RESULTADO DO RANDOM FOREST CALIBRADO PARA MÁXIMA PRECISÃO (LIMIAR > 0.75) ===
              precision    recall  f1-score   support

Baixa Evasão       0.74      0.98      0.84      1361
 Alta Evasão       0.68      0.12      0.20       543

    accuracy                           0.73      1904
   macro avg       0.71      0.55      0.52      1904
weighted avg       0.72      0.73      0.66      1904

Matriz de Confusão:
[[1332   29]
 [ 480   63]]
